# IOC and Artifact Derivation

> **Safety boundary:** This workbook performs static analysis of sanitized excerpts, fake fixtures, and derived public metadata. It does not execute attacker code, inspect the analyst's environment, or contact any network.

## Derivation Goal

An IOC list should be reproducible from public evidence without exposing raw
private captures. This workbook normalizes a reviewed catalog containing only
already-public indicators and sanitized artifact references.

Indicators are not verdicts. Shared hosting addresses, repository names, and
generic command fragments need context, time bounds, and corroboration.

In [ ]:
import csv
import io
import json
from pathlib import Path

catalog = json.loads(
    Path("../fixtures/public-artifact-catalog.json").read_text()
)
records = catalog["records"]

normalized = []
seen = set()
for record in records:
    key = (record["type"].lower(), str(record["value"]).strip())
    if key in seen:
        continue
    seen.add(key)
    normalized.append({
        "type": key[0],
        "value": key[1],
        "context": record["context"],
    })

normalized

In [ ]:
grouped = {}
for record in normalized:
    grouped.setdefault(record["type"], []).append(record)

{ioc_type: len(items) for ioc_type, items in sorted(grouped.items())}

## Public Indicator Classes

| Class | Defensive use | Caution |
|---|---|---|
| domains and URLs | proxy/DNS review, repository scanning | serverless infrastructure may be shared or ephemeral |
| direct IP and port | firewall, flow, process-network correlation | time-bound ownership and reuse matter |
| SHA-256 | exact artifact matching | hash covers only the captured byte sequence |
| lifecycle command | manifest and endpoint review | fragments can have benign uses |
| custom header | proxy or application log hunting | requires HTTP visibility |
| artifact reference | evidence navigation | not itself an IOC |

In [ ]:
buffer = io.StringIO()
writer = csv.DictWriter(buffer, fieldnames=["type", "value", "context"])
writer.writeheader()
writer.writerows(normalized)
csv_preview = buffer.getvalue()

json_preview = json.dumps(normalized, indent=2)
{"csv": csv_preview, "json": json_preview}

## Evidence Hygiene

The catalog includes the public stage hash, public endpoints, direct-IP IOC,
port, lifecycle command, request marker, repository identity, and references
to public evidence documents. It excludes environment values, credentials,
cookies, auth headers, and the raw stage body.

**Strongest supported claim:** these indicators occur in the preserved evidence
and public report.

**Not proven by an IOC match alone:** malicious intent for every appearance or
successful compromise of another host.

**Detection:** combine at least two dimensions, such as npm ancestry plus a
custom header, or direct-IP egress plus the stage hash.

**Confidence upgrade:** timestamped DNS/proxy/flow telemetry, file provenance,
and process ancestry tied to the matching indicator.